**Cell 00 · Cell index** — each cell is prefixed with `Cell NN · name` (also in cell metadata `cell_label`).

- **01 · Intro**
- **02 · Load & clean**
- **03 · Unified summary overview**
- **04 · Unified summary code**
- **05 · Summary plots overview**
- **06 · Summary plots code**
- **07 · Modeling target overview**
- **08 · Modeling target code**
- **09 · Scale predictors overview**
- **10 · Scale & modeling table code**
- **11 · BMI skew before/after scaling**
- **12 · Binary modeling overview**
- **13 · Logistic regression (fit & evaluation)**
- **14 · Random forest (fit & evaluation)**
- **15 · Model comparison**


**Cell 01 · Intro**

# Heart Disease Prediction Model

Pipeline for **BRFSS 2015** health indicators ([UCI / Kaggle heart-disease subset](https://archive.ics.uci.edu/dataset/891/cdc+diabetes+health+indicators)): load and clean **`diabetes_binary_health_indicators_BRFSS2015.csv`**, unified descriptive summary, exploratory plots, binary modeling target **`heart_disease_outcome`**, and a standardized modeling matrix **`df_model_scaled`**.

Each row is one survey respondent. **Target:** reported coronary heart disease or myocardial infarction (`HeartDiseaseorAttack` → **`heart_disease_outcome`**, 0/1). Predictors are remaining BRFSS health, behavior, and demographic fields.


In [ ]:
# Cell 02 · Load & clean
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

DATA_DIR = Path(".")
DATA_FILE = DATA_DIR / "diabetes_binary_health_indicators_BRFSS2015.csv"
COMPILED_DIR = DATA_DIR / "compiled"
COMPILED_DIR.mkdir(exist_ok=True)

COLUMN_RENAME = {
    "Diabetes_binary": "diabetes_binary",
    "HighBP": "high_bp",
    "HighChol": "high_chol",
    "CholCheck": "chol_check",
    "BMI": "bmi",
    "Smoker": "smoker",
    "Stroke": "stroke",
    "HeartDiseaseorAttack": "heart_disease_or_attack",
    "PhysActivity": "phys_activity",
    "Fruits": "fruits",
    "Veggies": "veggies",
    "HvyAlcoholConsump": "hvy_alcohol_consump",
    "AnyHealthcare": "any_healthcare",
    "NoDocbcCost": "no_docbc_cost",
    "GenHlth": "gen_hlth",
    "MentHlth": "ment_hlth",
    "PhysHlth": "phys_hlth",
    "DiffWalk": "diff_walk",
    "Sex": "sex",
    "Age": "age",
    "Education": "education",
    "Income": "income",
}

df_raw = pd.read_csv(DATA_FILE)
df_raw = df_raw.rename(columns=COLUMN_RENAME)

# BRFSS export: numeric 0/1 and coded ordinals; enforce numeric dtypes
for col in df_raw.columns:
    df_raw[col] = pd.to_numeric(df_raw[col], errors="coerce")

n_before = len(df_raw)
df_raw = df_raw.dropna(how="any")
if len(df_raw) < n_before:
    print(f"Dropped {n_before - len(df_raw):,} rows with missing values")

dup = int(df_raw.duplicated().sum())
if dup:
    print(f"Duplicate rows: {dup:,} (kept for BRFSS survey-weight fidelity)")

print(f"Loaded: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
print(f"Heart disease prevalence: {df_raw['heart_disease_or_attack'].mean():.3%}")

# Working frame for summaries (predictors + outcome; no extra encoding required)
df = df_raw.copy()
df.head()


**Cell 03 · Unified summary overview**

## Unified summary table

One row per column: data type, **measurement level** (BRFSS coding), counts, central tendency, spread, skew, and IQR outliers for **ratio** / **interval** fields only.


In [ ]:
# Cell 04 · Unified summary code
measurement_levels = {
    "diabetes_binary": "nominal (binary)",
    "high_bp": "nominal (binary)",
    "high_chol": "nominal (binary)",
    "chol_check": "nominal (binary)",
    "bmi": "ratio",
    "smoker": "nominal (binary)",
    "stroke": "nominal (binary)",
    "heart_disease_or_attack": "nominal (binary)",
    "phys_activity": "nominal (binary)",
    "fruits": "nominal (binary)",
    "veggies": "nominal (binary)",
    "hvy_alcohol_consump": "nominal (binary)",
    "any_healthcare": "nominal (binary)",
    "no_docbc_cost": "nominal (binary)",
    "gen_hlth": "ordinal",
    "ment_hlth": "ordinal",
    "phys_hlth": "ordinal",
    "diff_walk": "nominal (binary)",
    "sex": "nominal (binary)",
    "age": "ordinal",
    "education": "ordinal",
    "income": "ordinal",
}


def column_mode(series: pd.Series):
    modes = series.mode(dropna=True)
    return modes.iloc[0] if len(modes) else np.nan


def iqr_outlier_count(series: pd.Series) -> int:
    values = pd.to_numeric(series, errors="coerce").dropna()
    if len(values) < 4:
        return 0
    q1, q3 = values.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return int(((values < lower) | (values > upper)).sum())


rows = []
for col in df.columns:
    s = df[col]
    is_numeric = pd.api.types.is_numeric_dtype(s)
    level = measurement_levels.get(col, "nominal")
    distribution_stats = level in {"interval", "ratio"}

    rows.append(
        {
            "column": col,
            "data_type": str(s.dtype),
            "measurement_level": level,
            "count": int(s.count()),
            "missing": int(s.isna().sum()),
            "unique_values": int(s.nunique()),
            "mean": float(s.mean()) if is_numeric else np.nan,
            "median": float(s.median()) if is_numeric else np.nan,
            "mode": column_mode(s),
            "min": float(s.min()) if is_numeric else np.nan,
            "max": float(s.max()) if is_numeric else np.nan,
            "range": float(s.max() - s.min()) if is_numeric else np.nan,
            "skew": float(stats.skew(s.dropna(), bias=False))
            if distribution_stats and s.notna().sum() >= 3
            else np.nan,
            "iqr_outlier_count": iqr_outlier_count(s)
            if distribution_stats
            else np.nan,
        }
    )

unified_summary = pd.DataFrame(rows).set_index("column")
for c in ["mean", "median", "min", "max", "range", "skew"]:
    unified_summary[c] = unified_summary[c].round(3)
unified_summary["iqr_outlier_count"] = unified_summary["iqr_outlier_count"].astype("Int64")
unified_summary


**Cell 05 · Summary plots overview**

Key BRFSS distributions: **heart disease** outcome, **age** bracket by **sex**, **BMI**, and self-rated **general health**.


In [ ]:
# Cell 06 · Summary plots code
import matplotlib.pyplot as plt

AGE_LABELS = {
    1: "18–24",
    2: "25–29",
    3: "30–34",
    4: "35–39",
    5: "40–44",
    6: "45–49",
    7: "50–54",
    8: "55–59",
    9: "60–64",
    10: "65–69",
    11: "70–74",
    12: "75–79",
    13: "80+",
}
GEN_HLTH_LABELS = {
    1: "Excellent",
    2: "Very good",
    3: "Good",
    4: "Fair",
    5: "Poor",
}

fig = plt.figure(figsize=(14, 12))
grid = fig.add_gridspec(3, 2, height_ratios=[1.0, 1.0, 1.0], hspace=0.35, wspace=0.25)

ax_outcome = fig.add_subplot(grid[0, :])
outcome_counts = df_raw["heart_disease_or_attack"].value_counts().sort_index()
bars = ax_outcome.bar(
    ["No heart disease", "Heart disease or attack"],
    [outcome_counts.get(0, 0), outcome_counts.get(1, 0)],
    color=["#4C72B0", "#C44E52"],
    edgecolor="white",
)
ax_outcome.set_title("Heart Disease or Attack (BRFSS)")
ax_outcome.set_ylabel("Count")
ax_outcome.bar_label(bars, padding=3, fmt="%d")

sex_colors = {0: "#E15759", 1: "#4E79A7"}
sex_labels = {0: "Female", 1: "Male"}

ax_age = fig.add_subplot(grid[1, 0])
for sex_code, label in sex_labels.items():
    ages = df_raw.loc[df_raw["sex"] == sex_code, "age"].dropna()
    ax_age.hist(
        ages,
        bins=np.arange(0.5, 14.5, 1),
        alpha=0.65,
        label=f"{label} (n={len(ages):,})",
        color=sex_colors[sex_code],
        edgecolor="white",
    )
ax_age.set_title("Age bracket by sex (coded)")
ax_age.set_xlabel("Age code (1=18–24 … 13=80+)")
ax_age.set_ylabel("Frequency")
ax_age.legend(fontsize=8)

ax_bmi = fig.add_subplot(grid[1, 1])
for outcome, name, color in (
    (0, "No", "#4C72B0"),
    (1, "Yes", "#C44E52"),
):
    subset = df_raw.loc[df_raw["heart_disease_or_attack"] == outcome, "bmi"]
    ax_bmi.hist(subset, bins=30, alpha=0.55, label=f"{name} (n={len(subset):,})", color=color)
ax_bmi.set_title("BMI by heart disease status")
ax_bmi.set_xlabel("BMI")
ax_bmi.legend()

ax_gen = fig.add_subplot(grid[2, :])
gen_counts = df_raw["gen_hlth"].value_counts().sort_index()
x = [GEN_HLTH_LABELS.get(int(i), str(i)) for i in gen_counts.index]
bars = ax_gen.bar(x, gen_counts.values, color="#59A14F", edgecolor="white")
ax_gen.set_title("General health (GenHlth)")
ax_gen.set_ylabel("Count")
ax_gen.bar_label(bars, padding=3, fmt="%d")
ax_gen.tick_params(axis="x", rotation=15)

plt.suptitle("Key Feature Summary — BRFSS 2015", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()


**Cell 07 · Modeling target overview**

## Modeling target (binary)

**`heart_disease_outcome`** is the binary label for modeling (0 = no, 1 = yes coronary heart disease or myocardial infarction). The raw column **`heart_disease_or_attack`** is dropped from predictors to avoid leakage. All other BRFSS indicators remain as features.


In [ ]:
# Cell 08 · Modeling target code
from IPython.display import display

HEART_DISEASE_OUTCOME_COL = "heart_disease_outcome"

predictor_cols = [c for c in df.columns if c != "heart_disease_or_attack"]
df_model = df[predictor_cols].copy()
df_model[HEART_DISEASE_OUTCOME_COL] = df_raw["heart_disease_or_attack"].astype(int)

outcome_reference = pd.DataFrame(
    {
        "class_id": [0, 1],
        "label": ["No heart disease or attack", "Heart disease or attack"],
    }
)

print(f"Modeling table: {df_model.shape[0]:,} rows × {df_model.shape[1]} columns")
print(f"Outcome column: {HEART_DISEASE_OUTCOME_COL}")
display(outcome_reference)
print("\nClass counts:")
print(df_model[HEART_DISEASE_OUTCOME_COL].value_counts().sort_index().to_string())
df_model.head()


**Cell 09 · Scale predictors overview**

### Standardize continuous predictors

**`StandardScaler`** is fit on **ratio** columns only (**`bmi`**). Ordinal BRFSS codes (age, education, income, general health, mental/physical health days) and binary indicators stay on their original scales. Output: **`df_model_scaled`** for downstream classifiers.


In [ ]:
# Cell 10 · Scale & modeling table code
from IPython.display import display

from sklearn.preprocessing import StandardScaler

if "df_model" not in globals():
    raise NameError("Run Cell 08 (modeling target) first.")
if "measurement_levels" not in globals():
    raise NameError("Run Cell 04 (unified summary) first.")

CONTINUOUS_LEVELS = {"ratio", "interval"}

MODELING_PREDICTOR_COLS = [c for c in df_model.columns if c != HEART_DISEASE_OUTCOME_COL]
df_model_modeling = df_model[MODELING_PREDICTOR_COLS + [HEART_DISEASE_OUTCOME_COL]].copy()

CONTINUOUS_PREDICTOR_COLS = [
    c
    for c in MODELING_PREDICTOR_COLS
    if measurement_levels.get(c) in CONTINUOUS_LEVELS
]
ORDINAL_OR_NOMINAL_COLS = [
    c for c in MODELING_PREDICTOR_COLS if c not in CONTINUOUS_PREDICTOR_COLS
]

df_model_scaled = df_model_modeling.copy()
hd_modeling_scaler = StandardScaler()
if CONTINUOUS_PREDICTOR_COLS:
    df_model_scaled[CONTINUOUS_PREDICTOR_COLS] = hd_modeling_scaler.fit_transform(
        df_model_modeling[CONTINUOUS_PREDICTOR_COLS].astype(float)
    )

print(
    f"Modeling table: {df_model_modeling.shape[0]:,} rows × {df_model_modeling.shape[1]} columns "
    f"({len(MODELING_PREDICTOR_COLS)} predictors + {HEART_DISEASE_OUTCOME_COL})"
)
print(f"Continuous columns scaled ({len(CONTINUOUS_PREDICTOR_COLS)}): {CONTINUOUS_PREDICTOR_COLS}")
print(f"Unscaled predictors ({len(ORDINAL_OR_NOMINAL_COLS)}): {ORDINAL_OR_NOMINAL_COLS}")


def _count_iqr_outliers(series: pd.Series) -> int:
    if "iqr_outlier_count" in globals():
        return int(iqr_outlier_count(series))
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return int(((series < lower) | (series > upper)).sum())


def _final_modeling_feature_summary(frame: pd.DataFrame, features: list[str]) -> pd.DataFrame:
    rows: list[dict] = []
    for col in features:
        s = pd.to_numeric(frame[col], errors="coerce")
        vals = s.dropna()
        level = measurement_levels.get(col, "unknown")
        distribution_stats = level in {"interval", "ratio"}
        rows.append(
            {
                "feature": col,
                "measurement_level": level,
                "scaled": col in CONTINUOUS_PREDICTOR_COLS,
                "count": int(vals.count()),
                "mean": round(float(vals.mean()), 4) if len(vals) else None,
                "std": round(float(vals.std(ddof=0)), 4) if len(vals) else None,
                "min": round(float(vals.min()), 4) if len(vals) else None,
                "max": round(float(vals.max()), 4) if len(vals) else None,
                "skew": round(float(vals.skew()), 3) if len(vals) >= 3 else None,
                "iqr_outliers": _count_iqr_outliers(vals)
                if distribution_stats and len(vals) >= 4
                else None,
            }
        )
    return pd.DataFrame(rows).set_index("feature")


print("\n=== Final modeling matrix — feature summary ===")
display(_final_modeling_feature_summary(df_model_scaled, MODELING_PREDICTOR_COLS))

print("\n=== Final modeling matrix — first rows ===")
display(df_model_scaled.head(10))

export_path = COMPILED_DIR / "brfss_heart_disease_modeling_table.parquet"
try:
    df_model_scaled.to_parquet(export_path, index=False)
    print(f"\nSaved: {export_path}")
except Exception as exc:
    csv_path = COMPILED_DIR / "brfss_heart_disease_modeling_table.csv"
    df_model_scaled.to_csv(csv_path, index=False)
    print(f"\nParquet unavailable ({exc}); saved CSV: {csv_path}")


In [ ]:
# Cell 11 · BMI skew before and after StandardScaler
import matplotlib.pyplot as plt
from scipy import stats

if "df_model_modeling" not in globals() or "df_model_scaled" not in globals():
    raise NameError("Run Cell 10 (scale & modeling table) first.")

bmi_before = df_model_modeling["bmi"].astype(float)
bmi_after = df_model_scaled["bmi"].astype(float)
skew_before = float(stats.skew(bmi_before, bias=False))
skew_after = float(stats.skew(bmi_after, bias=False))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
axes[0].hist(bmi_before, bins=60, color="#4C72B0", edgecolor="white")
axes[0].set_title(f"BMI — before scaling\n(skew = {skew_before:.3f})")
axes[0].set_xlabel("BMI (kg/m²)")
axes[0].set_ylabel("Count")

axes[1].hist(bmi_after, bins=60, color="#59A14F", edgecolor="white")
axes[1].set_title(f"BMI — after StandardScaler\n(skew = {skew_after:.3f})")
axes[1].set_xlabel("Standardized BMI (z-score)")

fig.suptitle("Effect of scaling on BMI distribution", fontsize=13, y=1.02)
fig.tight_layout()
plt.show()

print(
    f"BMI skew: {skew_before:.4f} → {skew_after:.4f} "
    "(standardization centers and scales; skew magnitude typically changes)."
)


**Cell 12 · Binary modeling overview**

Predict **`heart_disease_outcome`** from **`df_model_scaled`** using a stratified **75/25** train/test split (`random_state=42`). Same evaluation as the respiratory notebook: **ROC AUC**, **confusion matrix**, **accuracy**, **precision**, **recall**, **F1**, and **`classification_report`**.

Random forest uses **`max_samples=50_000`** per tree for practical runtime on ~250k BRFSS rows (logistic regression trains on the full training split).


In [ ]:
# Cell 13 · Logistic regression — fit & evaluation
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    RocCurveDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split

if "df_model_scaled" not in globals():
    raise NameError("Run Cell 10 first.")

HD_MODEL_RANDOM_STATE = 42
HD_MODEL_TEST_SIZE = 0.25
HD_CLASS_NAMES = ["No heart disease", "Heart disease or attack"]

frame = df_model_scaled.copy()
X = frame[MODELING_PREDICTOR_COLS].astype(float)
y = frame[HEART_DISEASE_OUTCOME_COL].astype(int)
if X.isna().any().any():
    raise ValueError("Predictors contain NaN.")

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=HD_MODEL_TEST_SIZE,
    random_state=HD_MODEL_RANDOM_STATE,
    stratify=y,
)

hd_logistic_model = LogisticRegression(max_iter=5000, solver="lbfgs")
hd_logistic_model.fit(X_train, y_train)

hd_logistic_coef_table = pd.DataFrame(
    {
        "feature": MODELING_PREDICTOR_COLS,
        "coefficient": hd_logistic_model.coef_.ravel(),
    }
)
hd_logistic_coef_table["odds_ratio"] = np.exp(hd_logistic_coef_table["coefficient"])

print(
    f"Logistic regression — target={HEART_DISEASE_OUTCOME_COL}, "
    f"predictors={len(MODELING_PREDICTOR_COLS)}, train={len(y_train):,}, test={len(y_test):,}"
)
print(f"Intercept: {float(hd_logistic_model.intercept_[0]):.4f}")
display(
    hd_logistic_coef_table.sort_values(
        "coefficient", key=lambda s: s.abs(), ascending=False
    ).head(15)
)
def _hd_binary_classification_eval(
    y_true,
    y_pred,
    y_prob_positive,
    class_names: list[str],
    model_title: str,
    cm_cmap: str = "Blues",
) -> pd.Series:
    """ROC/AUC, confusion matrix, and metrics (binary)."""
    test_auc = roc_auc_score(y_true, y_prob_positive)
    print(f"{model_title} — test ROC AUC: {test_auc:.4f}")

    fig, ax = plt.subplots(figsize=(6, 5))
    RocCurveDisplay.from_predictions(
        y_true,
        y_prob_positive,
        ax=ax,
        name=f"{model_title} (test)",
    )
    ax.plot([0, 1], [0, 1], "k--", lw=0.8, label="Chance")
    ax.set_title(f"{model_title} — ROC (AUC = {test_auc:.3f})")
    ax.legend(loc="lower right")
    fig.tight_layout()
    plt.show()

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    print("Confusion matrix (rows = true, columns = predicted):")
    print(cm)

    fig_cm, ax_cm = plt.subplots(figsize=(4.8, 4.2))
    ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names).plot(
        ax=ax_cm,
        cmap=cm_cmap,
        colorbar=False,
    )
    ax_cm.set_title(f"{model_title} — confusion matrix (test, threshold=0.5)")
    fig_cm.tight_layout()
    plt.show()

    metrics = pd.Series(
        {
            "accuracy": accuracy_score(y_true, y_pred),
            "precision": precision_score(y_true, y_pred, zero_division=0),
            "recall": recall_score(y_true, y_pred, zero_division=0),
            "f1": f1_score(y_true, y_pred, zero_division=0),
            "roc_auc": test_auc,
        },
        name="value",
    )
    display(metrics.to_frame())
    print(classification_report(y_true, y_pred, target_names=class_names, digits=3))
    return metrics

y_prob_log = hd_logistic_model.predict_proba(X_test)[:, 1]
y_pred_log = hd_logistic_model.predict(X_test)
hd_logistic_test_metrics = _hd_binary_classification_eval(
    y_test,
    y_pred_log,
    y_prob_log,
    HD_CLASS_NAMES,
    "Logistic regression",
    cm_cmap="Blues",
)
hd_logistic_test_metrics.name = "logistic_regression"


In [ ]:
# Cell 14 · Random forest — fit & evaluation
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    RocCurveDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

if "X_train" not in globals():
    raise NameError("Run Cell 13 (logistic) first to create the train/test split.")

HD_RF_MAX_SAMPLES = 50_000

hd_rf_model = RandomForestClassifier(
    n_estimators=200,
    max_samples=HD_RF_MAX_SAMPLES,
    random_state=HD_MODEL_RANDOM_STATE,
    n_jobs=1,
)
hd_rf_model.fit(X_train, y_train)

hd_rf_importance = pd.DataFrame(
    {
        "feature": MODELING_PREDICTOR_COLS,
        "importance": hd_rf_model.feature_importances_,
    }
).sort_values("importance", ascending=False)

print(
    f"Random forest — target={HEART_DISEASE_OUTCOME_COL}, trees=200, "
    f"max_samples={HD_RF_MAX_SAMPLES:,}, train={len(y_train):,}, test={len(y_test):,}"
)
display(hd_rf_importance.head(15))
def _hd_binary_classification_eval(
    y_true,
    y_pred,
    y_prob_positive,
    class_names: list[str],
    model_title: str,
    cm_cmap: str = "Blues",
) -> pd.Series:
    """ROC/AUC, confusion matrix, and metrics (binary)."""
    test_auc = roc_auc_score(y_true, y_prob_positive)
    print(f"{model_title} — test ROC AUC: {test_auc:.4f}")

    fig, ax = plt.subplots(figsize=(6, 5))
    RocCurveDisplay.from_predictions(
        y_true,
        y_prob_positive,
        ax=ax,
        name=f"{model_title} (test)",
    )
    ax.plot([0, 1], [0, 1], "k--", lw=0.8, label="Chance")
    ax.set_title(f"{model_title} — ROC (AUC = {test_auc:.3f})")
    ax.legend(loc="lower right")
    fig.tight_layout()
    plt.show()

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    print("Confusion matrix (rows = true, columns = predicted):")
    print(cm)

    fig_cm, ax_cm = plt.subplots(figsize=(4.8, 4.2))
    ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names).plot(
        ax=ax_cm,
        cmap=cm_cmap,
        colorbar=False,
    )
    ax_cm.set_title(f"{model_title} — confusion matrix (test, threshold=0.5)")
    fig_cm.tight_layout()
    plt.show()

    metrics = pd.Series(
        {
            "accuracy": accuracy_score(y_true, y_pred),
            "precision": precision_score(y_true, y_pred, zero_division=0),
            "recall": recall_score(y_true, y_pred, zero_division=0),
            "f1": f1_score(y_true, y_pred, zero_division=0),
            "roc_auc": test_auc,
        },
        name="value",
    )
    display(metrics.to_frame())
    print(classification_report(y_true, y_pred, target_names=class_names, digits=3))
    return metrics

y_prob_rf = hd_rf_model.predict_proba(X_test)[:, 1]
y_pred_rf = hd_rf_model.predict(X_test)
hd_rf_test_metrics = _hd_binary_classification_eval(
    y_test,
    y_pred_rf,
    y_prob_rf,
    HD_CLASS_NAMES,
    "Random forest",
    cm_cmap="Greens",
)
hd_rf_test_metrics.name = "random_forest"


In [ ]:
# Cell 15 · Compare logistic vs random forest (test set)
from IPython.display import display

if "hd_logistic_test_metrics" not in globals() or "hd_rf_test_metrics" not in globals():
    raise NameError("Run Cells 13 and 14 first.")

hd_model_comparison = pd.DataFrame(
    {
        "logistic_regression": hd_logistic_test_metrics,
        "random_forest": hd_rf_test_metrics,
    }
)
hd_model_comparison["delta_rf_minus_logistic"] = (
    hd_model_comparison["random_forest"] - hd_model_comparison["logistic_regression"]
)

print(
    f"Test-set comparison — n={len(y_test):,}, "
    f"predictors={len(MODELING_PREDICTOR_COLS)}, random_state={HD_MODEL_RANDOM_STATE}"
)
display(hd_model_comparison.round(4))

auc_log = float(hd_logistic_test_metrics["roc_auc"])
auc_rf = float(hd_rf_test_metrics["roc_auc"])
if auc_rf > auc_log:
    winner = "random forest"
elif auc_rf < auc_log:
    winner = "logistic regression"
else:
    winner = "tie"
print(f"Higher ROC AUC on test: {winner} (logistic={auc_log:.4f}, RF={auc_rf:.4f}).")
